In [ ]:
%%bash
if [ -d "./cifar10_data/cifar-10-batches-py" ] && [ -f "./cifar10_data/cifar-10-batches-py/batches.meta" ]; then
    echo "Dataset already exists, skipping download."
else
    echo "Dataset not found, downloading..."
    kaggle datasets download -d pankrzysiu/cifar10-python --force
    unzip -q cifar10-python.zip -d ./cifar10_data
fi

In [ ]:
import pickle
import numpy as np
import os

def unpickle(file):
    with open(file, 'rb') as fo:
        data_dict = pickle.load(fo, encoding='bytes')
    return data_dict

data_dir = './cifar10_data/cifar-10-batches-py/'

x_train_list = []
y_train_list = []

for i in range(1, 6):
    batch_path = os.path.join(data_dir, f'data_batch_{i}')
    batch = unpickle(batch_path)
    x_train_list.append(batch[b'data'])
    y_train_list.append(batch[b'labels'])

x_train = np.concatenate(x_train_list, axis=0)
y_train = np.concatenate(y_train_list, axis=0)

test_batch = unpickle(os.path.join(data_dir, 'test_batch'))
x_test = test_batch[b'data']
y_test = np.array(test_batch[b'labels'])

meta_batch = unpickle(os.path.join(data_dir, 'batches.meta'))
classes = [b.decode('utf-8') for b in meta_batch[b'label_names']]

print(f"Training features shape: {x_train.shape}")
print(f"Training labels shape:   {y_train.shape}")
print(f"Testing features shape:  {x_test.shape}")
print(f"Dataset classes:         {classes}")

In [ ]:
from sklearn.model_selection import train_test_split

x_train_reshaped = x_train.reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1).astype('float32') / 255.0
x_test_reshaped = x_test.reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1).astype('float32') / 255.0

train_idx = np.where(y_train < 5)[0]
test_idx = np.where(y_test < 5)[0]

x_train_5 = x_train_reshaped[train_idx]
y_train_5 = y_train[train_idx]
x_test_5 = x_test_reshaped[test_idx]
y_test_5 = y_test[test_idx]

x_tr, x_val, y_tr, y_val = train_test_split(x_train_5, y_train_5, train_size=8000, test_size=1500, random_state=42, stratify=y_train_5)
x_te, _, y_te, _ = train_test_split(x_test_5, y_test_5, train_size=1500, random_state=42, stratify=y_test_5)

print("Train shape:", x_tr.shape, y_tr.shape)
print("Val shape:", x_val.shape, y_val.shape)
print("Test shape:", x_te.shape, y_te.shape)

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(factor=10/360)
])

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, BatchNormalization, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

train_dataset = tf.data.Dataset.from_tensor_slices((x_tr, y_tr))
train_dataset = train_dataset.shuffle(buffer_size=1024).batch(64)
train_dataset = train_dataset.map(
    lambda x, y: (data_augmentation(x, training=True), y),
    num_parallel_calls=tf.data.AUTOTUNE
).prefetch(buffer_size=tf.data.AUTOTUNE)

val_dataset = tf.data.Dataset.from_tensor_slices((x_val, y_val)).batch(64).prefetch(buffer_size=tf.data.AUTOTUNE)

model = Sequential([
    Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(32, 32, 3)),
    BatchNormalization(),
    MaxPooling2D((2, 2)),

    Conv2D(64, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(5, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    patience=3,
    factor=0.2,
    min_lr=1e-6
)

history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=30,
    callbacks=[early_stopping, reduce_lr]
)

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

y_pred_probs = model.predict(x_te)
y_pred = np.argmax(y_pred_probs, axis=1)

test_acc = accuracy_score(y_te, y_pred)
test_f1 = f1_score(y_te, y_pred, average='macro')

print("Test Accuracy:", test_acc)
print("Macro F1-Score:", test_f1)

target_names = classes[:5]
cm = confusion_matrix(y_te, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=target_names, yticklabels=target_names, cmap='Blues')
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()

plt.figure(figsize=(10, 4))
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Training and Validation Loss Curves')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:
model.save('my_cnn.h5')
reloaded_model = tf.keras.models.load_model('my_cnn.h5')

reloaded_pred_probs = reloaded_model.predict(x_te)
reloaded_pred = np.argmax(reloaded_pred_probs, axis=1)
reloaded_acc = accuracy_score(y_te, reloaded_pred)

print("Original Test Accuracy:", test_acc)
print("Reloaded Test Accuracy:", reloaded_acc)
print("Is accuracy identical?:", np.isclose(test_acc, reloaded_acc))

In [ ]:
random_indices = np.random.choice(len(x_te), size=5, replace=False)
fig, axes = plt.subplots(1, 5, figsize=(15, 3))

for idx, i in enumerate(random_indices):
    img = x_te[i]
    true_label_idx = int(y_te[i])
    true_label = classes[true_label_idx]

    pred_prob = reloaded_pred_probs[i]
    pred_label_idx = np.argmax(pred_prob)
    pred_label = classes[pred_label_idx]
    confidence = pred_prob[pred_label_idx] * 100.0

    print(f"Index {i}: True: {true_label}, Pred: {pred_label}, Confidence: {confidence:.2f}%")

    axes[idx].imshow(img)
    axes[idx].set_title(f"{pred_label}\n({confidence:.1f}%)")
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

### Reflection

#### 1. Why should augmentation be applied only to training data and not validation/test data?
Validation and test datasets are designed to measure the model's performance on realistic, unaltered data representing the target deployment distribution. Augmenting validation or test sets would modify the evaluation distribution, making accuracy metrics noisy or misleading, and failing to reflect actual performance on real-world test inputs.

#### 2. Why save the model after training instead of retraining every time?
Training deep learning models requires substantial computational resources and time. Saving the model architecture and trained weights allows us to instantly restore it for deployment or downstream fine-tuning in milliseconds, bypassing the redundant and expensive training step.

#### 3. What does a confidence score of 51% on a prediction tell you about the model?
In a 5-class classification task, a random guess has a $20\%$ chance of correctness. A $51\%$ confidence score indicates that while the model favors this class above the others, it remains highly uncertain. The input image likely contains ambiguous features, noise, or lies near the decision boundary in the representation space, making the prediction close to a coin toss.